In [1]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="min_direction_margin_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [2]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)



---[ TableVault Record ]---
device: mps
---[ TableVault Record ]---



In [3]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
model_name = "typeform/distilbert-base-uncased-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = {int(k): v for k, v in model.config.id2label.items()}
entailment_id = next(i for i, label in id2label.items() if "entail" in label.lower())
contradiction_id = next(i for i, label in id2label.items() if "contrad" in label.lower())

print("model:", model_name)
print("id2label:", id2label)
print("entailment_id:", entailment_id, "label:", id2label[entailment_id])
print("contradiction_id:", contradiction_id, "label:", id2label[contradiction_id])



---[ TableVault Record ]---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

model: typeform/distilbert-base-uncased-mnli
id2label: {0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}
entailment_id: 0 label: ENTAILMENT
contradiction_id: 2 label: CONTRADICTION
---[ TableVault Record ]---



In [5]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))



---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
---[ TableVault Record ]---



In [6]:
batch_size = 64
margin_12_all = []
margin_21_all = []
min_margin_all = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        enc_12 = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc_21 = tokenizer(
            batch_s2,
            batch_s1,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        enc_12 = {k: v.to(device) for k, v in enc_12.items()}
        enc_21 = {k: v.to(device) for k, v in enc_21.items()}

        logits_12 = model(**enc_12).logits
        logits_21 = model(**enc_21).logits

        margin_12 = (logits_12[:, entailment_id] - logits_12[:, contradiction_id]).detach().cpu().numpy()
        margin_21 = (logits_21[:, entailment_id] - logits_21[:, contradiction_id]).detach().cpu().numpy()
        min_margin = np.minimum(margin_12, margin_21)

        margin_12_all.extend(margin_12.tolist())
        margin_21_all.extend(margin_21.tolist())
        min_margin_all.extend(min_margin.tolist())

margin_12_all = np.array(margin_12_all)
margin_21_all = np.array(margin_21_all)
min_margin_all = np.array(min_margin_all)
y_pred = (min_margin_all > 0).astype(int)

print("done")
print("predicted_positive_rate:", float(y_pred.mean()))



---[ TableVault Record ]---


  0%|          | 0/7 [00:00<?, ?it/s]

done
predicted_positive_rate: 0.3235294117647059
---[ TableVault Record ]---



In [7]:
vault.create_record_list("mrpc_min_margin_prediction", column_names=["prediction", "min_margin"])

for i in range(len(y_pred)):
    vault.append_record("mrpc_min_margin_prediction", 
                        {
                            "prediction": int(y_pred[i]),
                            "min_margin": float(min_margin_all[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "mrpc_min_margin_prediction is an example-level prediction dataset derived from the GLUE MRPC validation split. For each sentence pair in glue_mrpc_validation, it stores the binary paraphrase prediction produced by a zero-shot NLI model (typeform/distilbert-base-uncased-mnli) and the confidence score used to make that prediction.\n\nEach record corresponds to one MRPC validation example and has two fields: prediction, an integer label where 1 indicates paraphrase and 0 indicates not_paraphrase; and min_margin, a float equal to the minimum of the entailment-versus-contradiction logit margins computed in both input directions, sentence1\u2192sentence2 and sentence2\u2192sentence1. The prediction is defined as 1 when min_margin > 0, otherwise 0.\n\nIn this workflow, this dataset serves as the per-example output of the min-direction-margin inference step. It is used as the input to downstream evaluation and summarization, including accuracy, F1, classification report generation, and error analysis."
embedding = get_embeddings(description)
vault.create_description("mrpc_min_margin_prediction", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "model predictions", "source_dataset": "glue_mrpc_validation", "source": "glue/mrpc", "split": "validation", "size": "408", "input_type": "sentence pair", "output_type": "binary prediction with confidence margin", "output_columns": "prediction,min_margin", "model": "typeform/distilbert-base-uncased-mnli", "method": "minimum directional entailment-contradiction margin", "label_space": "0=not_paraphrase,1=paraphrase"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_min_margin_prediction", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [8]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))



---[ TableVault Record ]---
{'accuracy': 0.5857843137254902, 'f1': 0.5888077858880778}
                precision    recall  f1-score   support

not_paraphrase       0.43      0.91      0.58       129
    paraphrase       0.92      0.43      0.59       279

      accuracy                           0.59       408
     macro avg       0.67      0.67      0.59       408
  weighted avg       0.76      0.59      0.59       408

---[ TableVault Record ]---



In [9]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("margin_12:", float(margin_12_all[i]))
    print("margin_21:", float(margin_21_all[i]))
    print("min_margin:", float(min_margin_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))



---[ TableVault Record ]---
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
margin_12: 7.967843055725098
margin_21: 6.533724784851074
min_margin: 6.533724784851074
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
margin_12: -9.776514053344727
margin_21: -0.8201401233673096
min_margin: -9.776514053344727
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
margin_12: -0.6154601573944092
marg

In [10]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("margin_12:", float(margin_12_all[i]))
    print("margin_21:", float(margin_21_all[i]))
    print("min_margin:", float(min_margin_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))



---[ TableVault Record ]---
num_errors: 169
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
margin_12: -1.0258152484893799
margin_21: 6.178720474243164
min_margin: -1.0258152484893799
true: 1 pred: 0
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
margin_12: -1.392435073852539
margin_21: -1.2508978843688965
min_margin: -1.392435073852539
true: 1 pred: 0
idx: 7
sentence1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
sentence2: IBM said the Rational products were also integrated with Rational PurifyPlus , which allows

In [11]:
vault.create_record_list("min_direction_margin_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("min_direction_margin_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "mrpc_min_margin_prediction": [0, len(ds)]
                    })

summary

description = "Summary dataset for the min_direction_margin_mrpc workflow. It contains experiment-level evaluation results for paraphrase detection on the GLUE MRPC validation set, where predictions are made by scoring each sentence pair in both directions with an MNLI model and using the minimum entailment-minus-contradiction margin as the decision signal. The dataset has one record with three fields: accuracy (float), f1 (float), and classification_report (string). Its role is to store the final aggregated performance summary derived from the source validation examples and the per-example prediction dataset mrpc_min_margin_prediction, so future users can quickly inspect the overall outcome of this notebook without recomputing metrics."
embedding = get_embeddings(description)
vault.create_description("min_direction_margin_mrpc_summary", description, embedding)

properties = {"artifact_type": "evaluation summary", "task": "paraphrase detection", "benchmark": "GLUE", "dataset": "MRPC", "source": "glue_mrpc_validation", "split": "validation", "domain": "news", "evaluated_examples": "408", "label_space": "binary", "model": "typeform/distilbert-base-uncased-mnli", "inference_method": "min directional entailment-contradiction margin", "prediction_source": "mrpc_min_margin_prediction", "metrics": "accuracy,f1,classification_report", "process": "min_direction_margin_mrpc", "framework": "transformers"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("min_direction_margin_mrpc_summary", cat, embedding, prop)




---[ TableVault Record ]---
---[ TableVault Record ]---



In [12]:
description = "This notebook evaluates a directional-margin approach to paraphrase detection on the GLUE MRPC validation set. It loads sentence pairs and labels from TableVault, runs the typeform/distilbert-base-uncased-mnli sequence classification model as an NLI-based paraphrase scorer in both directions (sentence1\u2192sentence2 and sentence2\u2192sentence1), computes the entailment-minus-contradiction logit margin for each direction, and uses the minimum of the two margins as the final paraphrase score; pairs with positive minimum margin are predicted as paraphrases. The workflow then stores per-example predictions and margins in TableVault, computes accuracy, F1, and a classification report against the MRPC labels, inspects sample predictions and errors, and saves a summary record and notebook-level metadata back to TableVault." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("min_direction_margin_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "method": "bidirectional NLI min-direction margin", "dataset": "glue/mrpc validation", "model": "typeform/distilbert-base-uncased-mnli", "model_family": "DistilBERT", "inference_type": "zero-shot sequence pair classification", "label_rule": "predict paraphrase when min(entailment-contradiction margin in both directions) > 0", "evaluation": "accuracy, f1-score, classification report", "frameworks": "PyTorch, Transformers, Hugging Face Datasets, scikit-learn", "storage": "TableVault", "metadata_embeddings": "OpenAI text-embedding-3-large", "process_name": "min_direction_margin_mrpc"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("min_direction_margin_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

